# Session 3 - Choosing a model without fooling yourself

**Solutions.** Worked code and the numbers you should expect.

## Choosing a model without fooling yourself

You now have a trustworthy way to score one model. Today you have to choose
between many, and tune each of them — which creates a new way to be wrong, one
that no amount of careful cross-validation inside a single model can fix.

**This session (2 h)**

1. A comparison table with error bars, and a paired comparison that respects folds
2. Hyperparameter search, and how much of a leaderboard is noise
3. Nested cross-validation: scoring the *procedure*, not the winner
4. Learning curves: should you measure more molecules, or better descriptors?

In [ ]:
import sys, warnings
sys.path.insert(0, "..")          # so that `import qsar_helpers` works
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from qsar_helpers import (load_fish, load_aquatic, xy, fold_error,
                          molecule_groups, score_table,
                          DESCRIPTOR_NOTES, TARGET_LABEL)

RNG = 0                            # one seed for the whole notebook
plt.rcParams.update({"figure.figsize": (6, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3, "font.size": 9})
print("pandas", pd.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (KFold, RepeatedKFold, GridSearchCV,
                                     cross_val_score, cross_validate, learning_curve)
from sklearn.base import clone
from sklearn.metrics import r2_score

fish = load_fish()
X, y = xy(fish)
cv_rep = RepeatedKFold(n_splits=5, n_repeats=5, random_state=RNG)   # 25 folds

## 1. The comparison table

### EXERCISE 3.1

Cross-validate six models with `cv_rep` (5-fold, 5 repeats = 25 folds):
dummy, linear regression, ridge (alpha=1), kNN (k=5), a random forest
(500 trees) and `HistGradientBoostingRegressor`. Scale where it matters.

Produce a table of mean R<sup>2</sup> ± sd and mean RMSE, and a horizontal bar
chart with error bars. Then answer: which models are distinguishable from each
other, given the spreads?

*Runtime about two minutes.*

In [ ]:
models = {
    "Dummy (mean)": DummyRegressor(strategy="mean"),
    "LinearRegression": make_pipeline(StandardScaler(), LinearRegression()),
    "Ridge (a=1)": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "kNN (k=5)": make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=5)),
    "RandomForest": RandomForestRegressor(n_estimators=500, random_state=RNG),
    "HistGB": HistGradientBoostingRegressor(random_state=RNG),
}

folds, rows = {}, []
for name, est in models.items():
    res = cross_validate(est, X, y, cv=cv_rep,
                         scoring=("r2", "neg_root_mean_squared_error"))
    folds[name] = res["test_r2"]
    rows.append(dict(model=name,
                     R2=res["test_r2"].mean(), R2_sd=res["test_r2"].std(),
                     RMSE=-res["test_neg_root_mean_squared_error"].mean()))
table = pd.DataFrame(rows).sort_values("R2", ascending=False)
print(table.round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 3.2))
t = table.iloc[::-1]
ax.barh(t.model, t.R2, xerr=t.R2_sd, color="#4C72B0", capsize=3)
ax.set_xlabel("cross-validated $R^2$  (mean $\\pm$ sd over 25 folds)")
ax.set_title("Six models, same 25 folds")
fig.tight_layout()

Dummy at -0.01, the two linear models tied at 0.56, and the three non-linear
models bunched between 0.60 and 0.61 — with fold-to-fold spreads of 0.05 to
0.06, i.e. larger than the gaps between them.

Two readings of that table are both correct and worth stating explicitly. The
step from "no model" to "linear model" is large and unambiguous. The step from
"linear" to "random forest" is about 0.05 in R<sup>2</sup>, which is the same
size as the error bars — so the bar chart alone cannot establish it.

### EXERCISE 3.2 — compare models on the folds they share

The fold-to-fold sd includes variation caused by the *folds* (some quarters of
this dataset are simply harder), and that variation is common to all models.
Because every model saw the same 25 folds, you can subtract it out: compute the
**per-fold difference** between two models instead of comparing their means.

1. Compute `folds["RandomForest"] - folds["Ridge (a=1)"]` and report its mean
   and sd. In what fraction of the 25 folds does the forest win?
2. Plot the paired fold scores (ridge on x, forest on y, with the diagonal) so
   the pairing is visible.
3. A paired t-test on these 25 differences would give a very small p-value.
   Explain why that p-value should not be reported.

In [ ]:
d = folds["RandomForest"] - folds["Ridge (a=1)"]
print(f"RF - Ridge, per fold: mean {d.mean():+.3f}   sd {d.std():.3f}")
print(f"forest wins in {100*(d>0).mean():.0f}% of the 25 folds")

fig, ax = plt.subplots(figsize=(4.2, 4.2))
lo, hi = 0.40, 0.75
ax.plot([lo, hi], [lo, hi], color="k", lw=1, ls="--")
ax.scatter(folds["Ridge (a=1)"], folds["RandomForest"], s=22, alpha=0.75, color="#4C72B0")
ax.set_xlabel("Ridge $R^2$"); ax.set_ylabel("RandomForest $R^2$")
ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
ax.set_title("Same fold, two models")
fig.tight_layout()

# (3) The 25 "observations" are not independent: the 5 folds within a repeat
#     share molecules by construction, and across repeats the same 908
#     molecules are reused. A paired t-test assumes independent draws, so its
#     standard error is far too small and the p-value is meaningless -- it can
#     be driven arbitrarily low simply by raising n_repeats, without collecting
#     a single new molecule. Report the effect size and the win fraction, and
#     be explicit that the comparison is on one dataset.

The forest is ahead by 0.049 ± 0.041 per fold and wins 84% of folds: a real but
modest improvement, consistently in the same direction. That sentence is
defensible. "The random forest substantially outperformed linear models" is not.

## 2. Hyperparameter search

### EXERCISE 3.3

Use `GridSearchCV` to tune k for kNN over `[1, 2, 3, 5, 8, 12, 20, 30, 50]`.
Plot mean CV R<sup>2</sup> against k with its error bars, and report the best k.

In [ ]:
ks = [1, 2, 3, 5, 8, 12, 20, 30, 50]
gs = GridSearchCV(make_pipeline(StandardScaler(), KNeighborsRegressor()),
                  {"kneighborsregressor__n_neighbors": ks},
                  scoring="r2", cv=KFold(5, shuffle=True, random_state=RNG)).fit(X, y)

m = gs.cv_results_["mean_test_score"]; s = gs.cv_results_["std_test_score"]
fig, ax = plt.subplots()
ax.errorbar(ks, m, yerr=s, marker="o", capsize=3, color="#4C72B0")
ax.set_xscale("log"); ax.set_xticks(ks); ax.set_xticklabels(ks)
ax.set_xlabel("k (neighbours)"); ax.set_ylabel("CV $R^2$")
ax.set_title("Validation curve: too flexible on the left, too rigid on the right")
fig.tight_layout()

print(f"best k = {gs.best_params_['kneighborsregressor__n_neighbors']}, "
      f"CV R2 = {gs.best_score_:.3f}")
print(pd.DataFrame({"k": ks, "R2": m.round(3), "sd": s.round(3)}).to_string(index=False))

k=1 is badly overfit (R<sup>2</sup> ≈ 0.41: a single nearest neighbour is mostly
measurement noise), the curve peaks broadly around k=8-12, and large k drifts
back toward the dummy. The peak is flat — which is the point of the next
exercise.

### EXERCISE 3.4 — how much of a leaderboard is noise?

Build a list of model configurations: kNN for each k above, ridge for
alpha in `[0.01, 0.1, 1, 10, 100, 1000]`, random forests crossing
`max_depth` in `[2, 3, 5, 8, None]` with `max_features` in `[0.3, 0.6, 1.0]`,
and `HistGradientBoostingRegressor` crossing `learning_rate` in
`[0.03, 0.1, 0.3]` with `max_leaf_nodes` in `[15, 31, 63]`. That is 39
configurations.

Score every one with the **same** 5-fold split, sort them, and print the top
eight. Then compare the gap between 1st and 8th place with the ±0.045 sd of a
single split you measured in session 2.

*Runtime two to three minutes.*

In [ ]:
cands = {}
for k in [1, 2, 3, 5, 8, 12, 20, 30, 50]:
    cands[f"kNN{k}"] = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=k))
for a in [0.01, 0.1, 1, 10, 100, 1000]:
    cands[f"ridge{a}"] = make_pipeline(StandardScaler(), Ridge(alpha=a))
for depth in [2, 3, 5, 8, None]:
    for mf in [0.3, 0.6, 1.0]:
        cands[f"rf_d{depth}_mf{mf}"] = RandomForestRegressor(
            n_estimators=200, max_depth=depth, max_features=mf, random_state=RNG)
for lr in [0.03, 0.1, 0.3]:
    for leaves in [15, 31, 63]:
        cands[f"hgb_lr{lr}_l{leaves}"] = HistGradientBoostingRegressor(
            learning_rate=lr, max_leaf_nodes=leaves, random_state=RNG)
print(len(cands), "configurations")

cv_shared = KFold(5, shuffle=True, random_state=RNG)
board = pd.Series({n: cross_val_score(m, X, y, cv=cv_shared, scoring="r2").mean()
                   for n, m in cands.items()}).sort_values(ascending=False)
print(board.head(8).round(3).to_string())
print(f"\n1st - 8th = {board.iloc[0]-board.iloc[7]:.3f}")
print("single-split sd measured in session 2 = 0.045")

The top eight configurations span 0.023 in R<sup>2</sup> — half the sd of a single
train/test split, and they come from three different model families. The
identity of the "winner" is therefore mostly a property of this particular fold
assignment, not of the models. Re-run it with a different `random_state` in
`cv_shared` and a different configuration will be on top.

Two practical consequences. Reporting *only* the winner's score overstates what
you know. And the effort of squeezing out the last 0.02 is better spent on
better descriptors or more molecules — which is exactly what the learning curve
at the end of this session will tell you.

## 3. Nested cross-validation

The score above is contaminated in a way session 2 did not cover. We used the
same folds to *choose* the configuration and to *report* its performance. The
maximum of 39 noisy numbers is biased upward — the winner is partly whichever
configuration best fits that fold assignment's quirks.

Nested CV measures the whole procedure "try all 39, keep the best" instead:

- an **outer** split sets aside a test fold;
- within the remaining data, an **inner** CV picks the configuration;
- the winner is refit on the outer training part and scored once on the outer
  test fold;
- repeat for every outer fold.

### EXERCISE 3.5

Implement nested CV over the 39 configurations (5 outer folds, 5 inner folds)
and compare it with the flat "best score on the leaderboard" number.

*This fits 39 x 5 x 5 models and takes several minutes — start it, then read on.*

In [ ]:
outer = KFold(5, shuffle=True, random_state=7)
inner = KFold(5, shuffle=True, random_state=1)
nested, picked = [], []
for tr, te in outer.split(X):
    scores = {n: cross_val_score(m, X.iloc[tr], y[tr], cv=inner, scoring="r2").mean()
              for n, m in cands.items()}
    win = max(scores, key=scores.get)
    picked.append(win)
    best = clone(cands[win]).fit(X.iloc[tr], y[tr])
    nested.append(r2_score(y[te], best.predict(X.iloc[te])))
nested = np.array(nested)

print("configuration chosen in each outer fold:", picked)
print(f"nested CV R2            = {nested.mean():.3f} +- {nested.std():.3f}")
print(f"flat leaderboard winner = {board.iloc[0]:.3f}")
print(f"selection optimism      = {board.iloc[0]-nested.mean():+.3f}")

At n=908 the measured optimism is about **-0.01** — not detectable, and in fact
the nested estimate (0.645 ± 0.048) comes out slightly *above* the flat winner
(0.631). With a fold-to-fold sd of 0.048 that difference is noise; the honest
reading is that selection bias here is below what this experiment can resolve.

Worth noting what the outer folds chose: four of the five picked
`rf(max_depth=None, max_features=0.3)` and the fifth picked the same forest with
`max_features=0.6`. So the *family* is selected stably even though the
leaderboard positions within it are not — a useful distinction when someone
claims their architecture search found something.

### EXERCISE 3.6 — where selection bias does bite

Selection bias grows with the number of configurations tried and shrinks with
sample size. Repeat 3.5 on random subsamples of **120 molecules**, three times
with different subsamples, and report the mean optimism.

*Reduce the forests to `n_estimators=100` to keep this under a few minutes.*

In [ ]:
small = {n: (clone(m).set_params(n_estimators=100)
             if isinstance(m, RandomForestRegressor) else m)
         for n, m in cands.items()}

rg = np.random.default_rng(1)
gaps = []
for rep in range(3):
    ii = rg.choice(len(X), 120, replace=False)
    Xs, ys = X.iloc[ii], y[ii]
    flat = max(cross_val_score(m, Xs, ys, cv=KFold(5, shuffle=True, random_state=rep),
                               scoring="r2").mean() for m in small.values())
    nest = []
    for tr, te in KFold(5, shuffle=True, random_state=100 + rep).split(Xs):
        sc = {n: cross_val_score(m, Xs.iloc[tr], ys[tr],
                                 cv=KFold(5, shuffle=True, random_state=1), scoring="r2").mean()
              for n, m in small.items()}
        w = clone(small[max(sc, key=sc.get)]).fit(Xs.iloc[tr], ys[tr])
        nest.append(r2_score(ys[te], w.predict(Xs.iloc[te])))
    gaps.append(flat - np.mean(nest))
    print(f"rep {rep}: flat best {flat:.3f} | nested {np.mean(nest):.3f} | optimism {gaps[-1]:+.3f}")
print(f"\nmean optimism at n=120: {np.mean(gaps):+.3f}   (at n=908 it was about -0.01)")

At 120 molecules the same procedure is optimistic by **+0.062 on average**
(+0.126, +0.030, +0.029 across the three subsamples). Same code, same 39
configurations — only the sample size changed. Note also how much the three
subsamples differ: at this n, the optimism is itself a noisy quantity, and a
single estimate of it would not be worth much either.

The general rule: the inflation scales with the number of things you tried
divided by the amount of data you tried them on. Reported performance from a
small cohort after a large architecture or hyperparameter search should be read
as an upper bound, unless the paper says it nested the selection.

## 4. Learning curves: what to spend money on

A practical question with a real budget behind it. The model is wrong by a
factor of about eight in concentration. To improve it, should the lab measure
more molecules, or invest in better descriptors?

### EXERCISE 3.7

Use `learning_curve` for the random forest and for ridge regression, with
training sizes from 10% to 100%. Plot training and cross-validated R<sup>2</sup>
against training-set size for both. Then answer:

1. Which model is limited by variance (overfitting), and which by bias?
2. Is the forest's cross-validated score still rising at n=726?
3. What would you advise the lab to do?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.8), sharey=True)
for ax, (name, est) in zip(axes, [
        ("RandomForest", RandomForestRegressor(n_estimators=200, random_state=RNG)),
        ("Ridge", make_pipeline(StandardScaler(), Ridge(alpha=1.0)))]):
    sizes, tr_s, te_s = learning_curve(est, X, y, train_sizes=np.linspace(0.1, 1.0, 8),
                                       cv=KFold(5, shuffle=True, random_state=RNG),
                                       scoring="r2")
    ax.plot(sizes, tr_s.mean(1), marker="o", color="#C44E52", label="training score")
    ax.plot(sizes, te_s.mean(1), marker="s", color="#4C72B0", label="cross-validated")
    ax.fill_between(sizes, te_s.mean(1) - te_s.std(1), te_s.mean(1) + te_s.std(1),
                    alpha=0.2, color="#4C72B0")
    ax.set_title(name); ax.set_xlabel("training molecules")
    print(f"{name}: train {np.round(tr_s.mean(1), 3)}")
    print(f"{name}: cv    {np.round(te_s.mean(1), 3)}")
axes[0].set_ylabel("$R^2$"); axes[0].legend(loc="center right")
fig.tight_layout()

The two curves say opposite things.

The **forest** sits at 0.94 on its training data and 0.60 in cross-validation.
That 0.34 gap is variance: the model is fitting molecule-specific noise it
cannot generalise. Its cross-validated score is still climbing at the full
dataset (0.52 at n=72 to 0.60 at n=726, with no plateau), so more molecules
would still help this model.

**Ridge** scores 0.58 on training data and 0.56 in cross-validation. There is
essentially no gap, and the curve flattens by about n=400. More molecules will
not help it at all: it is bias-limited, and the ceiling is set by what a linear
function of these six descriptors can express.

So the honest advice depends on which model you intend to deploy — and for the
flexible one, *both* more molecules and better descriptors would pay. The
descriptor question is the one session 4 opens up: six numbers cannot distinguish
molecules that differ in ways that matter for toxicity, which is also why 217 of
our rows have an identical twin.

## Session 3 checklist

1. Report every candidate's score, not only the winner's.
2. Compare models on shared folds, as paired differences; report effect size and
   win fraction, not a p-value from reused folds.
3. If you selected among configurations, either nest the selection or label the
   number an upper bound.
4. State the number of configurations you tried and the sample size, so a reader
   can judge the selection bias themselves.
5. Use a learning curve before asking for more data; it may not be the
   constraint.